# TickerLayer tutorial outputs

These cells read the real API responses captured on October 12, 2026. They do not request current prices. See the article for the live-request examples.

In [1]:
import json
from pathlib import Path
import pandas as pd
from market_monitor import quote_row

capture_path = Path("fresh_results/monitor_20261012T145742Z.json")
capture = json.loads(capture_path.read_text(encoding="utf-8"))
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 6)

In [2]:
monitor = pd.DataFrame([
    quote_row(item["asset"], item["payload"])
    for item in capture["quotes"]
])
monitor

,asset,symbol,bid,ask,mid,spread,event_time_utc
0,crypto,BTCUSD,8.259870e+04,8.259871e+04,8.259871e+04,0.01000,2026-10-12 14:57:26.856000+00:00
1,forex,EURUSD,1.117169e+00,1.117219e+00,1.117194e+00,0.00005,2026-10-12 14:57:26.004000+00:00
2,stocks,US:AAPL,3.410000e+02,3.410200e+02,3.410100e+02,0.02000,2026-10-12 14:57:27.151000+00:00
3,indices,AR40,2.827320e+06,2.828734e+06,2.828027e+06,1414.02000,2026-10-09 20:04:59+00:00
4,etfs,AAA,2.485000e+01,2.497000e+01,2.491000e+01,0.12000,2026-10-12 14:57:00.225000+00:00
5,commodities,XAUUSD,4.163762e+03,4.164182e+03,4.163972e+03,0.42000,2026-10-12 14:57:28.597000+00:00
6,perpetuals,AAOIUSDT,1.016650e+02,1.017550e+02,1.017100e+02,0.09000,2026-10-12 14:57:28.740000+00:00


In [3]:
bars = pd.DataFrame(capture["history"]["results"])
bars["date"] = pd.to_datetime(bars["t"], unit="ms", utc=True)
bars[["date", "o", "h", "l", "c", "v"]]

,date,o,h,l,c,v
0,2026-10-05 00:00:00+00:00,86530.00,86999.11,84972.01,85766.87,16186.77610
1,2026-10-06 00:00:00+00:00,85766.87,86698.99,85136.11,85549.93,12614.47572
2,2026-10-07 00:00:00+00:00,85549.94,85598.22,82787.25,83321.81,20838.08376
3,2026-10-08 00:00:00+00:00,83321.81,83521.15,80393.56,81754.45,21676.96164
4,2026-10-09 00:00:00+00:00,81754.46,83528.98,81603.52,82635.55,12301.61263
5,2026-10-10 00:00:00+00:00,82635.56,83173.29,82545.82,82988.36,7437.72181
6,2026-10-11 00:00:00+00:00,82988.35,84199.00,82804.65,83773.12,6963.51092


In [4]:
events = capture["websocket"]["events"]
quote_event = next(
    event["message"] for event in events
    if event["message"].get("type") == "quote"
)
print(json.dumps(quote_event, indent=2))

{
  "type": "quote",
  "channel": "crypto.quotes",
  "asset": "crypto",
  "symbol": "BTCUSD",
  "bid": "82614",
  "ask": "82614.01",
  "bid_size": "5.48359",
  "ask_size": "1.18694",
  "ts": 1791817054963
}


In [5]:
from collections import Counter

stream = capture["websocket"]
message_counts = Counter(
    event["message"].get("type", "unknown")
    for event in stream["events"]
)
print("Messages received:", len(stream["events"]))
print("Message types:", dict(message_counts))
print("Elapsed capture time:", stream["duration_seconds"], "seconds")

Messages received: 80
Message types: {'system': 3, 'trade': 62, 'quote': 15}
Elapsed capture time: 8.66 seconds


In [6]:
call = next(
    step for step in capture["mcp"]["steps"]
    if step["name"] == "tools_call_get_snapshot"
)
snapshot = call["body"]["result"]["structuredContent"]
print(json.dumps(snapshot, indent=2))

{
  "symbol": "BTCUSD",
  "bid": 82614,
  "ask": 82614.01,
  "bid_size": 4.84013,
  "ask_size": 1.33757,
  "last_price": 82521.68,
  "last_size": 0.00062074,
  "last_timestamp": 1791817063007,
  "prev_close": 83773.12,
  "change": -1251.44,
  "change_percent": -1.4938
}
